# Movie Recommendation System Using Hybrid Filtering

This notebook runs the project code over MovieLens latest-small. It demonstrates genre-based similarity, user-user collaborative filtering, and a hybrid ranking that combines both with a popularity prior. MovieLens user IDs are anonymized; they are examples, not actual accounts.

## 1. Locate the project

Run locally from the project folder, or open this notebook in Colab after uploading the project to a public GitHub repository. If the Python file is not found, this cell asks for the repository URL and clones it into the Colab runtime.

In [ ]:
from pathlib import Path
import subprocess
import sys

project_dir = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'movie_recommender.py').exists():
        project_dir = candidate
        break

if project_dir is None:
    repo_url = input('Paste your public movie-recommendation-system GitHub URL: ').strip()
    project_dir = Path('/content/movie-recommendation-system')
    subprocess.run(['git', 'clone', repo_url, str(project_dir)], check=True)

sys.path.insert(0, str(project_dir))
print(f'Project folder: {project_dir}')

## 2. Load ratings and build the recommender

The default example uses anonymized MovieLens user ID 1. You can replace it with another ID found in `ratings.csv`.

In [ ]:
import pandas as pd
from IPython.display import display
from movie_recommender import MovieRecommendationSystem, load_dataset, evaluate_system

movies, ratings = load_dataset()
recommender = MovieRecommendationSystem(movies, ratings)
print(f'{ratings.userId.nunique():,} users, {movies.movieId.nunique():,} movies, {len(ratings):,} ratings')

## 3. Personalized hybrid recommendations

The hybrid ranking weights collaborative preference at 60%, genre affinity at 30%, and a smoothed popularity prior at 10%. Titles already rated by the selected user are excluded.

In [ ]:
user_id = 1
recommendations = recommender.recommend_for_user(user_id, top_n=10, method='hybrid')
display(recommendations)

## 4. Similar-title recommendations

This content-based mode compares genre vectors and does not use user ratings.

In [ ]:
display(recommender.similar_movies('Toy Story (1995)', top_n=10))

## 5. Offline evaluation

The evaluation holds out 20% of each user's ratings using seed 42. It compares collaborative rating estimates with a global-mean baseline and checks the hybrid top-10 list against held-out ratings of at least four stars. Offline scores are not the same as real-world satisfaction.

In [ ]:
evaluation = evaluate_system(movies, ratings)
print('Rating prediction:')
display(pd.DataFrame(evaluation['rating_prediction']).T.round(3))
print('Hybrid ranking:')
display(pd.DataFrame([evaluation['hybrid_ranking']]).round(3))

## Dataset citation and limits

Harper, F. M., & Konstan, J. A. (2015). *The MovieLens Datasets: History and Context*. ACM Transactions on Interactive Intelligent Systems, 5(4), 19:1-19:19. <https://doi.org/10.1145/2827872>.

MovieLens latest-small is a development dataset generated in 2018. Its user IDs are anonymized, and it has no demographic profiles. Results from this sample do not establish that a person will enjoy a recommended movie.